# Smoke test ACB — seed 51

Notebook này kiểm tra nhanh pipeline `env_v3` sau khi thêm action mask, reward theo exposure, epsilon-UCB warm-up, Q-scale bonus và beta schedule theo coverage/transition. Nó không thay thế lượt chạy công bố 45 episode trên toàn bộ dữ liệu.

Phạm vi smoke: ACB, seed 51, GOOD/BAD, UCB-VAE/Epsilon-Greedy, dữ liệu và số update rút gọn.

In [ ]:
from __future__ import annotations

import json
from pathlib import Path

def find_main_notebook() -> Path:
    cwd = Path.cwd().resolve()
    relative = Path("application/EIDT_Project/Test_Three_Strategy_6Stocks.ipynb")
    candidates = [cwd / relative, cwd / "SARSA_FinancialRL" / relative,
                  Path("/kaggle/working/SARSA_FinancialRL") / relative,
                  *[parent / relative for parent in cwd.parents]]
    for candidate in candidates:
        if candidate.exists(): return candidate.resolve()
    raise FileNotFoundError("Không tìm thấy Test_Three_Strategy_6Stocks.ipynb")

MAIN_NOTEBOOK = find_main_notebook()
main = json.loads(MAIN_NOTEBOOK.read_text(encoding="utf-8"))
markers = ("# CELL 1", "# CELL 2", "# CELL 3", "# CELL 4", "# CELL 5")
sources = {}
for cell in main["cells"]:
    source = "".join(cell.get("source", []))
    for marker in markers:
        if source.startswith(marker): sources[marker] = source
missing = set(markers) - set(sources)
if missing: raise RuntimeError(f"Thiếu cell trong notebook chính: {sorted(missing)}")

scope = {"__name__": "acb_seed51_smoke"}
for marker in markers:
    source = sources[marker]
    if marker == "# CELL 5": source = source.split("\nif RUN_TRAINING:", 1)[0]
    exec(compile(source, str(MAIN_NOTEBOOK), "exec"), scope)
    if marker == "# CELL 1":
        scope.update({"TICKER": "ACB", "SEEDS": (51,), "CURRENT_RUN_SEED": 51,
                      "RUN_TRAINING": False, "RESUME": False})

globals().update(scope)
assert TICKER == "ACB" and SEEDS == (51,) and CURRENT_RUN_SEED == 51
assert RESULT_SCHEMA_VERSION == "env_v3_ucb_warmup_scaled_bonus"
print({"main_notebook": str(MAIN_NOTEBOOK), "ticker": TICKER, "seed": CURRENT_RUN_SEED,
       "schema": RESULT_SCHEMA_VERSION, "device": str(DEVICE)})

In [ ]:
# Cấu hình smoke: đủ đi qua warm-up/Q-update/evaluation nhưng không dùng để báo cáo khoa học.
SMOKE_EPISODES = 3
SMOKE_TRAIN_ROWS = 160
SMOKE_TEST_ROWS = 80
EPISODES = SMOKE_EPISODES
scope["EPISODES"] = SMOKE_EPISODES
MODEL_CONFIGS["UNCERTAINTY_AWARE_UCB_009"].update({
    "bootstrap_trajectories": 1,
    "bootstrap_updates": 2,
    "online_aux_updates": 1,
    "vae_batch_size": 64,
})
SMOKE_ROOT = OUTPUT_ROOT / "smoke_acb_seed51_env_v3"
SMOKE_ROOT.mkdir(parents=True, exist_ok=True)
print({"episodes": EPISODES, "train_rows": SMOKE_TRAIN_ROWS,
       "test_rows": SMOKE_TEST_ROWS, "output": str(SMOKE_ROOT)})

In [ ]:
metric_rows, curve_frames = [], []
for period in PERIODS:
    full_train, full_test = load_period_data("ACB", period)
    train = full_train.tail(SMOKE_TRAIN_ROWS).reset_index(drop=True)
    test = full_test.head(SMOKE_TEST_ROWS).reset_index(drop=True)
    scaler = fit_frozen_scaler(train)
    for model_key in MODEL_KEYS:
        set_seed(51)
        metric, curves, _, _, _ = run_one_seed(
            "ACB", period, model_key, train, test, scaler)
        metric_rows.append(metric)
        curve_frames.append(pd.DataFrame(curves))

smoke_metrics = pd.DataFrame(metric_rows)
smoke_curves = pd.concat(curve_frames, ignore_index=True)
smoke_metrics.to_csv(SMOKE_ROOT / "smoke_metrics.csv", index=False)
smoke_curves.to_csv(SMOKE_ROOT / "smoke_episode_curves.csv", index=False)

assert len(smoke_metrics) == len(PERIODS) * len(MODEL_KEYS)
assert smoke_metrics["finite"].all()
assert (smoke_metrics["test_invalid_action_rate"] == 0.0).all()
assert (smoke_metrics["train_invalid_action_rate"] == 0.0).all()
ucb = smoke_metrics[smoke_metrics["strategy"] == "ucb"]
assert (ucb["final_action_coverage"] > 0.0).all()
assert (ucb["final_ucb_epsilon"] > 0.0).all()
assert np.isfinite(ucb["test_q_scale_mean"]).all()

columns = ["period", "strategy", "seed", "test_profit", "test_sharpe",
           "test_collapse", "test_trade_count", "test_invalid_action_rate",
           "final_beta", "final_ucb_epsilon", "final_action_coverage",
           "test_q_scale_mean", "test_exploration_bonus_max"]
display(smoke_metrics[columns])
print("SMOKE TEST PASSED — ACB seed 51")